<a href="https://colab.research.google.com/github/LiquidVillain170/LiquidVillain170/blob/main/01_05_16_37_%E2%84%962_%D0%9D%D0%B0_GPU_T4_%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22spirinsalco_%D0%BE%D1%81%D0%BD%D0%BE%D0%B2%D0%B0_%D0%A5%D1%83%D0%B4%D0%BE%D0%B6%D0%BD%D0%B8%D0%BA%D0%B0_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install llama-cpp-python --extra-index-url https://github.io


Looking in indexes: https://pypi.org/simple, https://github.io
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.21-py3-none-linux_x86_64.whl size=18292428 sha256=01ec672cc74d81452b625281732c5057ae3f417697c2918245b84eef173d7281
  Stored in directory: /root/.cache/pip/wheels/8d/9c/44/39cb3a9e47ced67e64197053dace3cec12faf53daf81cac6c4
Successfully built llama-cpp-python


In [3]:
import IPython
from google.colab import output, drive
from llama_cpp import Llama
import time

# 1. МОНТИРОВАНИЕ ДИСКА
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Диск синхронизирован.")
except:
    print("⚠️ Ошибка диска. Убедись, что доступ разрешен.")

# 2. АКТИВАЦИЯ ЯДРА В GPU (Всё в видеопамять)
model_path = "/content/drive/MyDrive/ART_GEN_ENGINE/Llama-3-8B-Instruct-v0.1.Q4_K_M.gguf"

print("⚡ Загрузка Консильери в GPU T4... Подожди минуту.")
try:
    llm = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_gpu_layers=-1, # Полный перенос в видеопамять
        verbose=False
    )
    print("🚀 Ядро активировано. Видеопамять захвачена.")
except Exception as e:
    print(f"❌ Сбой загрузки ядра: {e}")

# 3. БЭКЕНД ЛОГИКА
def run_llama(text):
    # Промпт для Llama 3 (Instruct формат)
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nТы — элитный русский ИТ-аналитик. Исправляй ошибки, пиши эффективный код. Отвечай строго на русском.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

    try:
        res = llm(prompt, max_tokens=1024, stop=["<|eot_id|>"])
        # Безопасное извлечение текста
        if isinstance(res, dict) and 'choices' in res:
            answer = res['choices'][0]['text'].strip()
        else:
            answer = res['choices'][0]['text'].strip() # Для разных версий библиотеки
        return IPython.display.PlainText(answer)
    except Exception as e:
        return IPython.display.PlainText(f"Ошибка генерации: {str(e)}")

output.register_callback('run_llama', run_llama)

# 4. ФРОНТЕНД "ХОРОШИЙ БЛОКНОТ"
html_ui = """
<div style="background: #1a1a1a; color: #e0e0e0; padding: 25px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; border: 1px solid #333; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
    <h2 style="color: #007acc; margin-top: 0; border-bottom: 1px solid #333; padding-bottom: 10px;">🛡 ТЕРМИНАЛ КОНСИЛЬЕРИ</h2>

    <div id="display" style="height: 350px; overflow-y: auto; background: #000; padding: 15px; border-radius: 6px; border: 1px solid #222; margin-bottom: 15px; font-family: 'Consolas', monospace; font-size: 13px; line-height: 1.6;">
        <p style="color: #57a64a;">// Система в режиме GPU T4. Ожидание первого слонопотама...</p>
    </div>

    <textarea id="cmd" style="width: 100%; height: 100px; background: #252526; color: #fff; border: 1px solid #444; border-radius: 6px; padding: 12px; font-family: 'Consolas', monospace; outline: none;" placeholder="Вставь код или задай вопрос..."></textarea>

    <div style="display: flex; justify-content: space-between; align-items: center; margin-top: 15px;">
        <button onclick="send()" style="background: #007acc; color: white; border: none; padding: 12px 30px; border-radius: 6px; cursor: pointer; font-weight: bold; transition: 0.3s;">АНАЛИЗИРОВАТЬ</button>
        <span style="color: #666; font-size: 11px;">v1.0 | GPU Active</span>
    </div>
</div>

<script>
    async function send() {
        const cmd = document.getElementById('cmd');
        const disp = document.getElementById('display');
        const text = cmd.value.trim();
        if (!text) return;

        disp.innerHTML += `<div style="color: #9cdcfe; margin-top: 15px;"><b>➤ ВВОД:</b></div><div style="background: #1e1e1e; padding: 8px; border-left: 3px solid #007acc; margin: 5px 0;">${text}</div>`;
        disp.innerHTML += `<p id="loading" style="color: #dcdcaa;">⚡ Консильери препарирует код...</p>`;
        disp.scrollTop = disp.scrollHeight;

        cmd.value = '';

        try {
            const result = await google.colab.kernel.invokeFunction('run_llama', [text], {});
            document.getElementById('loading').remove();
            disp.innerHTML += `<div style="color: #4ec9b0; margin-top: 15px;"><b>❖ ОТВЕТ:</b></div><div style="white-space: pre-wrap; color: #d4d4d4; background: #1e1e1e; padding: 10px; border-radius: 4px;">${result.data['text/plain']}</div>`;
        } catch (e) {
            document.getElementById('loading').innerHTML = '<span style="color: #f44747;">❌ Ошибка связи с ядром. Перезапусти ячейку.</span>';
        }

        disp.scrollTop = disp.scrollHeight;
    }
</script>
"""
display(IPython.display.HTML(html_ui))


Mounted at /content/drive
✅ Диск синхронизирован.
⚡ Загрузка Консильери в GPU T4... Подожди минуту.


llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


🚀 Ядро активировано. Видеопамять захвачена.


/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1256: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/tmp/ipykernel_1323/3492667661.py", line 40, in run_llama
    return IPython.display.PlainText(answer)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'IPython.display' has no attribute 'PlainText'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2764, in user_expressions
    value = self._format_user_obj(eval(expr, global_ns, user_ns))
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/google/colab/output/_js.py", line 114, in _invoke_function
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1323/3492667661.py", line 42, in run_llama
    return IPython.display.PlainText(f"Ошибка генерации: {str(e)}")
           ^^^^^^^^^^^^^^^^^^^^^^^^^
Attrib

код для помощника на ПК

In [ ]:
import os
import socket
import platform
import shutil
from datetime import datetime

def log_event(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}")

def check_network_gateways(targets):
    """Проверка доступности шлюзов и портов"""
    log_event("Начинаю сканирование шлюзов...")
    for host, port in targets.items():
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(2)
        result = sock.connect_ex((host, port))
        if result == 0:
            log_event(f"✅ {host}:{port} — ШЛЮЗ ОТКРЫТ")
        else:
            log_event(f"❌ {host}:{port} — ШЛЮЗ ЗАКРЫТ (Error: {result})")
        sock.close()

def deep_clean():
    """Наведение порядка в системном мусоре"""
    log_event("Начинаю глубокую очистку...")

    # Список путей для Windows и Linux
    paths = []
    if platform.system() == "Windows":
        paths = [
            os.environ.get('TEMP'),
            os.path.join(os.environ.get('SystemRoot', 'C:\\Windows'), 'Temp'),
            os.path.join(os.environ.get('LOCALAPPDATA', ''), 'Google\\Chrome\\User Data\\Default\\Cache')
        ]
    else:
        paths = ['/tmp', '/var/tmp', os.path.expanduser('~/.cache')]

    for path in paths:
        if path and os.path.exists(path):
            log_event(f"Очистка директории: {path}")
            try:
                # Очистка содержимого без удаления самой папки
                for item in os.listdir(path):
                    item_path = os.path.join(path, item)
                    try:
                        if os.path.isfile(item_path): os.unlink(item_path)
                        elif os.path.isdir(item_path): shutil.rmtree(item_path)
                    except: continue
            except Exception as e:
                log_event(f"Ошибка доступа к {path}: {e}")

if __name__ == "__main__":
    # Настройка: IP и Порт (80 - http, 443 - https, 53 - dns)
    my_gateways = {
        "8.8.8.8": 53,
        "192.168.1.1": 80,
        "1.1.1.1": 443
    }

    check_network_gateways(my_gateways)
    deep_clean()
    log_event("Работа завершена. Порядок наведен.")
